# Machine Failure Prediction using Machine Learning


In [ ]:
#importing the libraries
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

## 1. Data Collection/loading dataset

This project aims to build machine learning models to predict whether a machine is likely to fail or not based on its operating conditions and sensor measurements.

The dataset contains multiple features such as air temperature, process temperature, rotational speed, torque, tool wear, and machine type.

This is a classification problem where:
- 0 = no machine failure
- 1 = machine failure

In [ ]:
# Load dataset
df = pd.read_csv("ai 2020.csv")

# Show first 5 rows
df.head()

## 2. Initial Data Validation

Before building the machine learning model, the dataset is checked to understand its structure and quality.

The checks include:
- dataset size
- column names
- summary statistics
- missing values
- duplicate rows
- target class distribution

This step is important because machine learning models require clean and correctly structured data.

In [ ]:
# Check the number of rows and columns
df.shape

In [ ]:
# Display all column names in the dataset
df.columns 

In [ ]:
# Show summary statistics for numerical columns
df.describe()

In [ ]:
# Check how many missing values are in each column
df.isnull().sum()

In [ ]:
# Check how many duplicated rows are in the dataset
df.duplicated().sum()

In [ ]:
# Check how many examples there are for each target class
df["Machine failure"].value_counts() 

In [ ]:
# Check the percentage of each class in the target column
df["Machine failure"].value_counts(normalize=True) * 100 

In [ ]:
#drop cdollums that are not significant/should not be used for prediction
df_drop = df.drop(columns=[
    "UDI", 
    "Product ID",
    "TWF", 
    "HDF", 
    "PWF", 
    "OSF", 
    "RNF"
])

## 3. Exploratory Data Analysis

EDA is used to understand the dataset before building the model.

The following are visualised:

- Target variable distribution
- Machine type distribution
- Correlation heatmap
- Histograms for each numerical feature 
- Boxplots for each numerical feature


In [ ]:
# Plot the distribution of the target variable
plt.figure(figsize=(6, 4))
sns.countplot(data=df, x="Machine failure")

plt.title("Distribution of Machine Failure")
plt.xlabel("Machine Failure")
plt.ylabel("Count")
plt.show()

In [ ]:
# Create a figure for the machine type distribution chart
plt.figure(figsize=(6, 4))
sns.countplot(data=df, x="Type")
plt.title("Machine Type Distribution")
plt.xlabel("Type")
plt.ylabel("Count")
plt.show()

In [ ]:
# Create a figure for the correlation heatmap
plt.figure(figsize=(10, 6))
sns.heatmap(df_drop.corr(numeric_only=True), annot=True, cmap="coolwarm")
plt.title("Correlation Heatmap")
plt.show()

In [ ]:
# Create a list of numerical features to visualise
numeric_features = [
    "Air temperature [K]",
    "Process temperature [K]",
    "Rotational speed [rpm]",
    "Torque [Nm]",
    "Tool wear [min]"
]
# Loop through each numerical feature
for col in numeric_features:
# Create a new figure for each histogram
    plt.figure(figsize=(6,4))
    sns.histplot(df[col], kde=True)
    plt.title(f"Distribution of {col}")
    plt.show()

In [ ]:
for col in numeric_features:
# Create a new figure for each boxplot
    plt.figure(figsize=(6,4))
    sns.boxplot(data=df, x="Machine failure", y=col)
    plt.title(f"{col} by Machine Failure")
    plt.show()

## 4. Data Preprocessing 
To prepare the data for modeling:
- We separate features (X) and target variable (y) 
- use OneHotEncoder is to convert categorical text data into numbers so that a machine learning model can use it.
- Numerical columns are scaled using MinMaxScaler

In [ ]:
# Separate the dataset into input features and target variable
X = df_drop.drop(columns=["Machine failure"])
y = df_drop["Machine failure"]

In [ ]:
encoder = OneHotEncoder(sparse_output=False)
# Encode the Type column
encoded_type = encoder.fit_transform(X[["Type"]])

# Convert encoded values into a DataFrame
encoded_type_df = pd.DataFrame(
    encoded_type,
    columns=encoder.get_feature_names_out(["Type"]),
    index=X.index
)

# Drop the original categorical column
X = X.drop("Type", axis=1)

# Add the encoded columns back to X
X = pd.concat([X, encoded_type_df], axis=1)
X.head()


In [ ]:
# Split the data into training and temporary sets
X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=42,
    stratify=y
)

# Split the temporary set into validation and test sets
X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    random_state=42,
    stratify=y_temp
)

# Display the shapes of each dataset
print("Training set:", X_train.shape)
print("Validation set:", X_val.shape)
print("Test set:", X_test.shape)

In [ ]:
# Create scaler
scaler = MinMaxScaler()

# Fit only on training data
X_train_scaled = scaler.fit_transform(X_train)

# Transform validation and test data
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)


## 5. Model 1: Logistic Regression

The first model is Logistic Regression.

In [ ]:
# Create the Logistic Regression model
log_model = LogisticRegression(max_iter=1000)
# Train the Logistic Regression model using the scaled training data
log_model.fit(X_train, y_train)


In [ ]:
# Use the trained Logistic Regression model to make predictions
log_val_predictions = log_model.predict(X_val_scaled)

# Calculate validation accuracy
log_val_accuracy = accuracy_score(y_val, log_val_predictions)

print("Logistic Regression - Validation Results")
print("Validation Accuracy:", log_val_accuracy)

print("\nClassification Report:")
print(classification_report(y_val, log_val_predictions))

print("\nConfusion Matrix:")
print(confusion_matrix(y_val, log_val_predictions))

# Create confusion matrix
log_cm = confusion_matrix(y_val, log_val_predictions)

plt.figure(figsize=(6, 4))
sns.heatmap(log_cm, annot=True, fmt="d", cmap="Blues")
plt.title("Logistic Regression Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()

## 6. Model 2: Random forest Classifier

The second model is a Random forest Classifier.

In [ ]:
# Create the Random Forest model
forest_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)
# Train the Random Forest model using the scaled training data
forest_model.fit(X_train_scaled, y_train)

In [ ]:
# Make predictions on the validation data
forest_val_predictions = forest_model.predict(X_val_scaled)
# Calculate validation accuracy
forest_val_accuracy = accuracy_score(y_val, forest_val_predictions)

print("Random Forest - Validation Results")
print("Validation Accuracy:", forest_val_accuracy)

print("\nClassification Report:")
print(classification_report(y_val, forest_val_predictions))

print("\nConfusion Matrix:")
print(confusion_matrix(y_val, forest_val_predictions))

# Create confusion matrix
forest_cm = confusion_matrix(y_val, forest_val_predictions)

# Plot confusion matrix using a different colour
plt.figure(figsize=(6, 4))
sns.heatmap(forest_cm, annot=True, fmt="d", cmap="Greens")
plt.title("Random Forest Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()

Two models are compared using validation accuracy: 

-random forest model has been chosen as a better model

In [ ]:
# Create a comparison table for the models
comparison = pd.DataFrame({
    "Model": [
        "Logistic Regression",
        "Random Forest"
    ],
    "Validation Accuracy": [
        log_val_accuracy,
        forest_val_accuracy
    ]
})

# Display comparison table
comparison

## 7. Hyperparameter Optimisation

To improve the model that was chousen, we adjust key parameters: 
- Numer of tress 
- Max depth 
- Min samples split


In [ ]:
# Create the optimised Random Forest model
forest_model_optimised = RandomForestClassifier(
    n_estimators=150,
    max_depth= None,
    min_samples_split= 2,
    random_state=42
)
# Train the optimised Random Forest model
forest_model_optimised.fit(X_train_scaled, y_train)

In [ ]:
# Make predictions on the validation data
forest_optimised_val_predictions = forest_model_optimised.predict(X_val_scaled)

# Calculate validation accuracy
forest_optimised_val_accuracy = accuracy_score(y_val, forest_optimised_val_predictions)

print("Optimised Random Forest - Validation Results")
print("Validation Accuracy:", forest_optimised_val_accuracy)

print("\nClassification Report:")
print(classification_report(y_val, forest_optimised_val_predictions))

print("\nConfusion Matrix:")
print(confusion_matrix(y_val, forest_optimised_val_predictions))

# Create confusion matrix
forest_optimised_cm = confusion_matrix(y_val, forest_optimised_val_predictions)

plt.figure(figsize=(6, 4))
sns.heatmap(forest_optimised_cm, annot=True, fmt="d", cmap="Purples")
plt.title("Optimised Random Forest Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()

In [ ]:
# Create model comparison table
comparison = pd.DataFrame({
    "Model": [
        "Logistic Regression",
        "Random Forest",
        "Optimised Random Forest"
    ],
    "Validation Accuracy": [
        log_val_accuracy,
        forest_val_accuracy,
        forest_optimised_val_accuracy
    ]
})
comparison

## 8. Testing on Previously Unseen Data

The final selected model is now tested on the unseen test set.

In [ ]:
# Take one sample from test data as DataFrame
sample = X_test.iloc[[0]]
# Show the sample features
print("Sample data:")
print(sample)
# Predict using tuned Random Forest
prediction = forest_model_optimised.predict(sample)
# Show predicted and actual class
print("Predicted class:", prediction[0])
print("Actual class:", y_test.iloc[0])

## 9. Conclusion

This project built a machine learning pipeline to predict machine failure.

The project included:

- Data collection
- Data validation
- Exploratory data analysis
- Feature preparation
- Training two machine learning models
- Hyperparameter optimisation
- Model comparison
- Testing on unseen data

The final model was selected based on validation performance, where Random Forest has achieved the best acuracy. 
Hyperparameter tuning provided improvement in the calidation performance for the random forest. Hence, 
The improved Random Forest model is selected as the best-performing model.